In [2]:
import json
from collections import defaultdict
from pathlib import Path

spam_path = Path("./PhishFuzzer_spam.jsonl")
valid_path = Path("./PhishFuzzer_valid.jsonl")
phishing_path = Path("./PhishFuzzer_phishing.jsonl")
output_path = Path("./PhishFuzzer_final.jsonl")

def sample_two_per_original_id(path: Path):
    groups = defaultdict(list)
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            row = json.loads(line)
            groups[row.get("Original_ID")].append(row)
    samples = []
    for original_id in sorted(groups.keys()):
        samples.extend(groups[original_id][:2])
    return samples

rows = []
rows.extend(sample_two_per_original_id(phishing_path))
rows.extend(sample_two_per_original_id(spam_path))
rows.extend(sample_two_per_original_id(valid_path))

with output_path.open("w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=True))
        f.write("\n")

print(f"Wrote {len(rows)} records to {output_path}")

Wrote 6600 records to PhishFuzzer_final.jsonl


In [8]:
import json
import random
from pathlib import Path

output_path = Path("./combined.jsonl")
seed = 42
target_per_class = 6000

random.seed(seed)


def read_jsonl(path: Path):
    rows = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            rows.append(json.loads(line))

    return rows


def sample_rows(rows, n):
    if len(rows) < n:
        raise ValueError(
            f"Not enough rows to sample: {len(rows)} available, {n} requested"
        )

    return random.sample(rows, n)


def relabel(rows, label_value):
    for r in rows:
        r["label"] = label_value

    return rows


# =========================================
# LABEL MAPPING
# =========================================
# 0 -> valid
# 1 -> spam
# 2 -> phishing
# =========================================


# =========================
# Spam sources
# =========================
spam_ceas = [
    r
    for r in read_jsonl(Path("./CEAS_08_sanitized_with_urls.jsonl"))
    if r.get("label") == 1
]

spam_pf = [
    r
    for r in read_jsonl(Path("./PhishFuzzer_final.jsonl"))
    if str(r.get("Type", "")).strip().lower() == "spam"
]

spam_pool = spam_ceas + spam_pf

# relabel spam as 1
spam_rows = relabel(
    sample_rows(spam_pool, target_per_class),
    1
)


# =========================
# Phishing sources
# =========================
phish_nf = [
    r
    for r in read_jsonl(Path("./Nigerian_Fraud_sanitized.jsonl"))
    if r.get("label") == 1
]

phish_pf = [
    r
    for r in read_jsonl(Path("./PhishFuzzer_final.jsonl"))
    if str(r.get("Type", "")).strip().lower() == "phishing"
]

phish_nz = [
    r
    for r in read_jsonl(Path("./Nazatio_sanitized.jsonl"))
    if r.get("label") == 1
]

phish_pool = phish_nf + phish_pf + phish_nz

# relabel phishing as 2
phish_rows = relabel(
    sample_rows(phish_pool, target_per_class),
    2
)


# =========================
# Valid sources
# =========================
valid_ceas = [
    r
    for r in read_jsonl(Path("./CEAS_08_sanitized_with_urls.jsonl"))
    if r.get("label") == 0
]

valid_pf = [
    r
    for r in read_jsonl(Path("./PhishFuzzer_final.jsonl"))
    if str(r.get("Type", "")).strip().lower() == "valid"
]

valid_legit = [
    r
    for r in read_jsonl(Path("./phishing_legitimate1.jsonl"))
    if r.get("label") == 0
]

valid_pool = valid_ceas + valid_pf + valid_legit

# relabel valid as 0
valid_rows = relabel(
    sample_rows(valid_pool, target_per_class),
    0
)


# =========================
# Combine + Shuffle
# =========================
combined = spam_rows + phish_rows + valid_rows

random.shuffle(combined)


# =========================
# Save
# =========================
with output_path.open("w", encoding="utf-8") as f:
    for r in combined:
        f.write(json.dumps(r, ensure_ascii=True))
        f.write("\n")


print(f"Wrote {len(combined)} records to {output_path}")

print("\nLabel Mapping:")
print("0 -> Valid")
print("1 -> Spam")
print("2 -> Phishing")

Wrote 18000 records to combined.jsonl

Label Mapping:
0 -> Valid
1 -> Spam
2 -> Phishing


In [10]:
import json
import re
from pathlib import Path

input_path = Path("./combined.jsonl")
output_path = Path("./combined_normalized.jsonl")

url_re = re.compile(r"(https?://[^\s<>\"]+)")
label_map = {
    0: "valid",
    1: "spam",
    2: "phishing",
}

def to_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [str(v) for v in value if str(v).strip()]
    if isinstance(value, str):
        value = value.strip()
        return [value] if value else []
    return []


def pick_first(row, keys):
    for key in keys:
        value = row.get(key)
        if value is not None and str(value).strip():
            return str(value)
    return ""


def collect_urls(row, body):
    urls = []
    for key in ("urls", "URL", "Url", "link", "links"):
        urls.extend(to_list(row.get(key)))
    if body:
        urls.extend(url_re.findall(body))
    seen = set()
    normalized = []
    for url in urls:
        url = url.strip()
        if not url:
            continue
        if url not in seen:
            seen.add(url)
            normalized.append(url)
    return normalized


def normalize_label(value):
    if value is None:
        return None
    if isinstance(value, str):
        value = value.strip()
        if value.isdigit():
            value = int(value)
    if isinstance(value, int):
        return label_map.get(value, value)
    return value


def normalize_row(row):
    subject = pick_first(row, ("subject", "Subject"))
    body = pick_first(row, ("body", "Body", "Text", "text"))
    urls = collect_urls(row, body)
    normalized = {
        "subject": subject,
        "body": body,
        "urls": urls,
    }
    if "label" in row:
        normalized["label"] = normalize_label(row.get("label"))
    return normalized


count = 0
with input_path.open("r", encoding="utf-8") as f_in, output_path.open("w", encoding="utf-8") as f_out:
    for line in f_in:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        normalized = normalize_row(row)
        f_out.write(json.dumps(normalized, ensure_ascii=True))
        f_out.write("\n")
        count += 1

print(f"Wrote {count} records to {output_path}")

Wrote 18000 records to combined_normalized.jsonl


In [13]:
import json
import re
from pathlib import Path
from html import unescape

INPUT_FILE = "combined_normalized.jsonl"
OUTPUT_FILE = "combined_cleaned.jsonl"


# ==========================================
# REGEX PATTERNS
# ==========================================

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
EMAIL_PATTERN = re.compile(r"\b[\w\.-]+@[\w\.-]+\.\w+\b")
PHONE_PATTERN = re.compile(r"\+?\d[\d\-\(\) ]{7,}\d")
NUMBER_PATTERN = re.compile(r"\b\d+\b")

HTML_TAG_PATTERN = re.compile(r"<.*?>")
MULTISPACE_PATTERN = re.compile(r"\s+")

LONG_SYMBOL_PATTERN = re.compile(r"[_=\-*#]{3,}")
NON_ASCII_CONTROL_PATTERN = re.compile(r"[\r\t]+")

REPEATED_CHAR_PATTERN = re.compile(r"(.)\1{5,}")




# ==========================================
# CLEANING FUNCTION
# ==========================================

def clean_text(text):

    if not text:
        return ""

    # Decode HTML entities
    text = unescape(text)

    # Remove HTML tags
    text = re.sub(HTML_TAG_PATTERN, " ", text)

    # Replace URLs
    text = re.sub(URL_PATTERN, " <URL> ", text)


    # Remove long symbol garbage
    text = re.sub(LONG_SYMBOL_PATTERN, " ", text)

    # Remove weird control chars
    text = re.sub(NON_ASCII_CONTROL_PATTERN, " ", text)

    # Remove repeated characters
    text = re.sub(REPEATED_CHAR_PATTERN, r"\1", text)

    # Remove excessive spaces
    text = re.sub(MULTISPACE_PATTERN, " ", text)

    return text.strip()




# ==========================================
# MAIN CLEANING LOOP
# ==========================================

cleaned_count = 0
removed_count = 0

with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    for line in fin:

        try:
            row = json.loads(line)

            subject = clean_text(row.get("subject", ""))
            body = clean_text(row.get("body", ""))

            # Skip empty emails
            if len(body.split()) < 5:
                removed_count += 1
                continue

            # Skip extremely long emails
            if len(body.split()) > 2000:
                removed_count += 1
                continue

            row["subject"] = subject
            row["body"] = body

            fout.write(json.dumps(row, ensure_ascii=False))
            fout.write("\n")

            cleaned_count += 1

        except Exception as e:
            removed_count += 1
            continue


print(f"Cleaned rows: {cleaned_count}")
print(f"Removed rows: {removed_count}")
print(f"Saved cleaned dataset to: {OUTPUT_FILE}")

Cleaned rows: 17702
Removed rows: 298
Saved cleaned dataset to: combined_cleaned.jsonl


In [15]:
import json
import numpy as np
from transformers import AutoTokenizer
from collections import defaultdict

# ==========================================
# CONFIG
# ==========================================

DATASET_PATH = "combined_cleaned.jsonl"

# Change this depending on model
MODEL_NAME = "distilbert-base-uncased"

# Example alternatives:
# MODEL_NAME = "Qwen/Qwen2.5-7B"
# MODEL_NAME = "mistralai/Mistral-7B-v0.1"



# ==========================================
# LOAD TOKENIZER
# ==========================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)



# ==========================================
# STORAGE
# ==========================================

all_lengths = []
label_lengths = defaultdict(list)

long_samples = []



# ==========================================
# READ DATASET
# ==========================================

with open(DATASET_PATH, "r", encoding="utf-8") as f:

    for idx, line in enumerate(f):

        row = json.loads(line)

        subject = row.get("subject", "")
        body = row.get("body", "")
        label = row.get("label", "unknown")

        # Combine subject + body
        text = f"Subject: {subject}\n\nBody: {body}"

        # Tokenize
        tokens = tokenizer.encode(
            text,
            truncation=False
        )

        token_length = len(tokens)

        all_lengths.append(token_length)
        label_lengths[label].append(token_length)

        # Store extremely long samples
        if token_length > 512:
            long_samples.append({
                "index": idx,
                "label": label,
                "tokens": token_length,
                "subject": subject[:100]
            })



# ==========================================
# GLOBAL STATS
# ==========================================

print("\n==============================")
print("GLOBAL TOKEN LENGTH STATS")
print("==============================")

print(f"Total Samples: {len(all_lengths)}")

print(f"Mean Tokens: {np.mean(all_lengths):.2f}")
print(f"Median Tokens: {np.median(all_lengths):.2f}")

print(f"Min Tokens: {np.min(all_lengths)}")
print(f"Max Tokens: {np.max(all_lengths)}")

print(f"95th Percentile: {np.percentile(all_lengths, 95):.2f}")
print(f"99th Percentile: {np.percentile(all_lengths, 99):.2f}")



# ==========================================
# MODEL LIMIT ANALYSIS
# ==========================================

print("\n==============================")
print("MODEL CONTEXT ANALYSIS")
print("==============================")

limits = [128, 256, 512, 1024, 2048]

for limit in limits:

    exceed = sum(1 for x in all_lengths if x > limit)

    percent = (exceed / len(all_lengths)) * 100

    print(
        f"Samples > {limit} tokens: "
        f"{exceed} ({percent:.2f}%)"
    )



# ==========================================
# CLASS-WISE ANALYSIS
# ==========================================

print("\n==============================")
print("CLASS-WISE TOKEN STATS")
print("==============================")

for label, lengths in label_lengths.items():

    print(f"\nLabel: {label}")

    print(f"Count: {len(lengths)}")

    print(f"Mean: {np.mean(lengths):.2f}")
    print(f"Median: {np.median(lengths):.2f}")

    print(f"95th Percentile: {np.percentile(lengths, 95):.2f}")

    print(f"Max: {np.max(lengths)}")



# ==========================================
# LONG SAMPLE ANALYSIS
# ==========================================

print("\n==============================")
print("LONG SAMPLES (>512 TOKENS)")
print("==============================")

print(f"Total Long Samples: {len(long_samples)}")

for sample in long_samples[:10]:

    print("\n------------------")

    print(f"Index: {sample['index']}")
    print(f"Label: {sample['label']}")
    print(f"Tokens: {sample['tokens']}")
    print(f"Subject: {sample['subject']}")



# ==========================================
# RECOMMENDATION
# ==========================================

print("\n==============================")
print("RECOMMENDATION")
print("==============================")

p95 = np.percentile(all_lengths, 95)

if p95 <= 256:
    print("Recommended max_length = 256")

elif p95 <= 512:
    print("Recommended max_length = 512")

elif p95 <= 1024:
    print("Recommended max_length = 1024")

else:
    print("Recommended max_length = 2048")


print("\nDone.")

/home/kunal/jhinga/fine-tune/fine-tune-email/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (757 > 512). Running this sequence through the model will result in indexing errors



GLOBAL TOKEN LENGTH STATS
Total Samples: 17702
Mean Tokens: 334.39
Median Tokens: 234.00
Min Tokens: 15
Max Tokens: 19537
95th Percentile: 862.95
99th Percentile: 1881.89

MODEL CONTEXT ANALYSIS
Samples > 128 tokens: 11852 (66.95%)
Samples > 256 tokens: 8322 (47.01%)
Samples > 512 tokens: 4060 (22.94%)
Samples > 1024 tokens: 600 (3.39%)
Samples > 2048 tokens: 132 (0.75%)

CLASS-WISE TOKEN STATS

Label: spam
Count: 5797
Mean: 180.57
Median: 84.00
95th Percentile: 617.00
Max: 1636

Label: phishing
Count: 5974
Mean: 405.82
Median: 346.00
95th Percentile: 812.35
Max: 19537

Label: valid
Count: 5931
Mean: 412.79
Median: 274.00
95th Percentile: 1302.00
Max: 5672

LONG SAMPLES (>512 TOKENS)
Total Long Samples: 4060

------------------
Index: 5
Label: valid
Tokens: 757
Subject: [UAI] Fuzzy logic and probability

------------------
Index: 6
Label: valid
Tokens: 550
Subject: [soaplite] Digest Number 1749

------------------
Index: 7
Label: valid
Tokens: 1473
Subject: [UAI] CALL FOR PAPERS CIMCA

In [16]:
import json
from transformers import AutoTokenizer

# ==========================================
# CONFIG
# ==========================================

INPUT_FILE = "combined_cleaned.jsonl"
OUTPUT_FILE = "combined_final.jsonl"

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

MAX_TOKENS = 2048


# ==========================================
# LOAD TOKENIZER
# ==========================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)


# ==========================================
# FILTER DATASET
# ==========================================

kept = 0
removed = 0

with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    for idx, line in enumerate(fin):

        try:
            row = json.loads(line)

            subject = row.get("subject", "")
            body = row.get("body", "")

            # Combine subject + body
            text = f"Subject: {subject}\n\nBody: {body}"

            # Tokenize
            tokens = tokenizer.encode(
                text,
                truncation=False
            )

            token_length = len(tokens)

            # Keep only samples <= MAX_TOKENS
            if token_length <= MAX_TOKENS:

                fout.write(
                    json.dumps(row, ensure_ascii=False)
                )
                fout.write("\n")

                kept += 1

            else:
                removed += 1

        except Exception as e:
            removed += 1
            continue


# ==========================================
# SUMMARY
# ==========================================

print("\n==========================")
print("FILTERING COMPLETE")
print("==========================")

print(f"Max Token Limit: {MAX_TOKENS}")

print(f"Kept Samples: {kept}")
print(f"Removed Samples: {removed}")

print(f"\nSaved filtered dataset to:")
print(OUTPUT_FILE)


FILTERING COMPLETE
Max Token Limit: 2048
Kept Samples: 17561
Removed Samples: 141

Saved filtered dataset to:
combined_final.jsonl


In [17]:
import json
import re
import math
from urllib.parse import urlparse
from collections import Counter

INPUT_FILE = "combined_final.jsonl"
OUTPUT_FILE = "combined_with_metadata.jsonl"


# ==========================================
# KEYWORD LISTS
# ==========================================

URGENCY_WORDS = [
    "urgent", "immediately", "verify", "suspend",
    "action required", "important", "alert",
    "warning", "expire", "limited"
]

MONEY_WORDS = [
    "bank", "invoice", "payment", "refund",
    "credit", "debit", "transaction", "cash",
    "salary", "bonus"
]

CREDENTIAL_WORDS = [
    "password", "login", "verify account",
    "credential", "authenticate", "otp",
    "security", "reset password"
]


# ==========================================
# REGEX
# ==========================================

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
IP_URL_PATTERN = re.compile(
    r"https?://(?:\d{1,3}\.){3}\d{1,3}"
)

ALL_CAPS_PATTERN = re.compile(r"\b[A-Z]{3,}\b")


# ==========================================
# HELPERS
# ==========================================

def shannon_entropy(text):

    if not text:
        return 0

    probs = [
        n_x / len(text)
        for x, n_x in Counter(text).items()
    ]

    return -sum(p * math.log2(p) for p in probs)


def count_keywords(text, keywords):

    text = text.lower()

    return sum(
        1 for word in keywords
        if word in text
    )


def extract_domains(urls):

    domains = []

    for url in urls:

        try:
            parsed = urlparse(url)

            domains.append(parsed.netloc.lower())

        except:
            continue

    return domains


# ==========================================
# PROCESS DATASET
# ==========================================

processed = 0

with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    for line in fin:

        try:
            row = json.loads(line)

            subject = row.get("subject", "")
            body = row.get("body", "")

            text = f"{subject} {body}"

            urls = URL_PATTERN.findall(text)

            domains = extract_domains(urls)

            # ==================================
            # METADATA FEATURES
            # ==================================

            metadata = {

                # Basic lengths
                "subject_length": len(subject),
                "body_length": len(body),

                "subject_word_count":
                    len(subject.split()),

                "body_word_count":
                    len(body.split()),

                # URL features
                "url_count": len(urls),

                "has_url":
                    len(urls) > 0,

                "has_ip_url":
                    bool(IP_URL_PATTERN.search(text)),

                "domains":
                    domains,

                # Text style features
                "exclamation_count":
                    text.count("!"),

                "question_count":
                    text.count("?"),

                "all_caps_word_count":
                    len(ALL_CAPS_PATTERN.findall(text)),

                # Security/social engineering features
                "urgency_keyword_count":
                    count_keywords(text, URGENCY_WORDS),

                "money_keyword_count":
                    count_keywords(text, MONEY_WORDS),

                "credential_keyword_count":
                    count_keywords(text, CREDENTIAL_WORDS),

                # Entropy
                "text_entropy":
                    round(shannon_entropy(text), 4),

                # Character ratios
                "digit_ratio":
                    sum(c.isdigit() for c in text)
                    / max(len(text), 1),

                "uppercase_ratio":
                    sum(c.isupper() for c in text)
                    / max(len(text), 1),
            }

            row["metadata"] = metadata

            fout.write(
                json.dumps(row, ensure_ascii=False)
            )

            fout.write("\n")

            processed += 1

        except Exception as e:
            continue


print("\n==========================")
print("METADATA EXTRACTION DONE")
print("==========================")

print(f"Processed Samples: {processed}")

print(f"\nSaved to:")
print(OUTPUT_FILE)


METADATA EXTRACTION DONE
Processed Samples: 17561

Saved to:
combined_with_metadata.jsonl


In [1]:
import json
from tqdm import tqdm

INPUT_FILE = "combined_with_metadata.jsonl"
OUTPUT_FILE = "instruction_dataset.jsonl"


# ==========================================
# SYSTEM PROMPT
# ==========================================

SYSTEM_PROMPT = (
    "You are an AI cybersecurity assistant trained to "
    "classify emails into spam, phishing, or valid categories."
)


# ==========================================
# COUNT TOTAL LINES FOR PROGRESS BAR
# ==========================================

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    total_lines = sum(1 for _ in f)


# ==========================================
# CREATE INSTRUCTION DATASET
# ==========================================

converted = 0
failed = 0

with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    for line in tqdm(
        fin,
        total=total_lines,
        desc="Converting Dataset",
        unit="samples"
    ):

        try:
            row = json.loads(line)

            subject = row.get("subject", "").strip()
            body = row.get("body", "").strip()

            label = row.get("label", "").strip()

            metadata = row.get("metadata", {})

            # Keep only raw email content
            email_text = (
                f"Subject: {subject}\n\n"
                f"Body:\n{body}"
            )

            # Instruction-style format
            formatted = {

                "messages": [

                    {
                        "role": "system",
                        "content": SYSTEM_PROMPT
                    },

                    {
                        "role": "user",
                        "content": email_text
                    },

                    {
                        "role": "assistant",
                        "content": label
                    }
                ],

                # Keep metadata separate
                "metadata": metadata
            }

            fout.write(
                json.dumps(formatted, ensure_ascii=False)
            )

            fout.write("\n")

            converted += 1

        except Exception as e:
            failed += 1
            continue


# ==========================================
# SUMMARY
# ==========================================

print("\n==============================")
print("INSTRUCTION DATASET CREATED")
print("==============================")

print(f"Converted Samples: {converted}")
print(f"Failed Samples: {failed}")

print(f"\nSaved to:")
print(OUTPUT_FILE)

Converting Dataset: 100%|██████████| 17561/17561 [00:00<00:00, 28804.56samples/s]


INSTRUCTION DATASET CREATED
Converted Samples: 17561
Failed Samples: 0

Saved to:
instruction_dataset.jsonl


In [2]:
import json
import hashlib
from tqdm import tqdm

INPUT_FILE = "instruction_dataset.jsonl"
OUTPUT_FILE = "instruction_dataset_dedup.jsonl"

seen_hashes = set()

kept = 0
removed = 0


def create_hash(messages):

    user_content = ""

    assistant_content = ""

    for m in messages:

        if m["role"] == "user":
            user_content = m["content"]

        elif m["role"] == "assistant":
            assistant_content = m["content"]

    combined = (
        user_content.strip().lower()
        + assistant_content.strip().lower()
    )

    return hashlib.md5(
        combined.encode("utf-8")
    ).hexdigest()


# Count lines
with open(INPUT_FILE, "r", encoding="utf-8") as f:
    total = sum(1 for _ in f)


with open(INPUT_FILE, "r", encoding="utf-8") as fin, \
     open(OUTPUT_FILE, "w", encoding="utf-8") as fout:

    for line in tqdm(fin, total=total):

        try:
            row = json.loads(line)

            sample_hash = create_hash(
                row["messages"]
            )

            if sample_hash not in seen_hashes:

                seen_hashes.add(sample_hash)

                fout.write(
                    json.dumps(row, ensure_ascii=False)
                )

                fout.write("\n")

                kept += 1

            else:
                removed += 1

        except:
            removed += 1


print("\n====================")
print("DEDUP COMPLETE")
print("====================")

print(f"Kept: {kept}")
print(f"Removed: {removed}")

100%|██████████| 17559/17559 [00:00<00:00, 24838.28it/s]


DEDUP COMPLETE
Kept: 16529
Removed: 1030
